# Geolocator

Works out where a photograph was taken from the picture itself — no GPS tag needed.

## Just do this

1. **Runtime › Run all** (or press `Ctrl+F9`), and approve the warning.
2. Wait about two minutes while it installs and loads the models.
3. Scroll to the **bottom** cell and upload your photo.

That is the whole thing — you do not need to run the cells one at a time.

> **Tip:** *Runtime › Change runtime type › T4 GPU* makes it several times faster, and is free.

---

### What to expect

| Kind of photo | Typical result |
|---|---|
| Street scene with readable signs | often the right city, sometimes the right street |
| Ordinary street, well-covered country | usually the right country and region |
| Generic nature: ocean, desert, plain forest | often only the right continent |

Every answer comes with an uncertainty radius. A wide radius means the photo genuinely
does not contain enough to narrow things down. That number is the part worth trusting.


## 1. Setup

Takes about two minutes. Downloads the CLIP backbone (~1.7 GB) the first time.


In [ ]:
%%capture
# Runtime > Change runtime type > T4 GPU makes this a few times faster,
# but it works fine on CPU too.
!pip install -q geoclip reverse_geocoder pycountry anthropic
!pip install -q --no-deps 'git+https://github.com/Oskar296/Oskar296.git@claude/location-guessing-ai-w2wzzx#subdirectory=geolocator'


## 2. API key

The models read the picture two ways. **Retrieval** matches it against 100,000 known
places and needs no key. **Reasoning** reads the signs, plates and road markings — that
is where most of the accuracy is, and it needs an Anthropic API key.

Get one at [console.anthropic.com](https://console.anthropic.com/settings/keys). The cell
below will ask you to paste it. Press Enter to skip and run retrieval-only.

To avoid pasting it every time: key icon in the left sidebar → **Add new secret** → name it
`ANTHROPIC_API_KEY` → paste the value → turn on **Notebook access**.


In [ ]:
import getpass
import os

key = os.environ.get('ANTHROPIC_API_KEY')

if not key:
    # A saved Colab secret is the no-friction path once it is set up.
    try:
        from google.colab import userdata
        key = userdata.get('ANTHROPIC_API_KEY')
    except Exception:
        key = None

if not key:
    print('No saved key found.')
    print('Paste your Anthropic API key, or press Enter to skip.')
    print('It stays in this session only and is not echoed.')
    key = getpass.getpass('Key: ').strip() or None

if key and not key.startswith('sk-ant-'):
    print(f"Warning: that does not look like an Anthropic key "
          f"(they start with 'sk-ant-'). Trying it anyway.")

if key:
    os.environ['ANTHROPIC_API_KEY'] = key
    print('Reasoning on. Both models will run.')
else:
    print('Retrieval only. Still works, but it cannot read signs.')


## 3. Load the models

One-off, about a minute. Leave the notebook running and you only pay this once.


In [ ]:
from geolocator import Geolocator
import torch

from geolocator.predictor import PredictorConfig
from geolocator.retrieval import RetrievalConfig

device = 'cuda' if torch.cuda.is_available() else 'cpu'
locator = Geolocator(PredictorConfig(retrieval=RetrievalConfig(device=device)))
locator.warm_up()
print(f'Ready, running on {device}.')


## 4. Upload your photo

This is the cell you actually use. Run it, choose an image, wait a few seconds.
Re-run it any time for another photo — the models stay loaded.


In [ ]:
from google.colab import files
from IPython.display import display, HTML
import folium


def show(path):
    pred = locator.locate(path)
    place = pred.place.describe() or 'Unnamed location'
    radius = (f'{pred.radius_km * 1000:,.0f} m' if pred.radius_km < 1
              else f'{pred.radius_km:,.0f} km')

    display(HTML(f'''
      <div style="font-family:system-ui;padding:14px 0">
        <div style="font-size:22px;font-weight:600">{place}</div>
        <div style="font-family:ui-monospace,monospace;color:#666">
          {pred.lat:.5f}, {pred.lon:.5f}</div>
        <div style="color:#666;margin-top:6px">within roughly {radius}
          &middot; {pred.confidence:.0%} confidence inside 200 km</div>
      </div>'''))

    m = folium.Map(location=[pred.lat, pred.lon], zoom_start=5)
    folium.Marker([pred.lat, pred.lon], tooltip=place).add_to(m)
    folium.Circle([pred.lat, pred.lon], radius=max(pred.radius_km, 1) * 1000,
                  color='#b4513a', fill=True, fill_opacity=0.08).add_to(m)
    for alt in pred.alternatives:
        folium.CircleMarker([alt.lat, alt.lon], radius=5, color='#888',
                            tooltip=f'{alt.label} ({alt.weight:.0%})').add_to(m)
    display(m)

    if pred.alternatives:
        print('Other possibilities:')
        for alt in pred.alternatives:
            print(f'  {alt.weight:5.1%}  {alt.label or f"{alt.lat:.2f}, {alt.lon:.2f}"}')

    for head in pred.heads:
        cues = (head.evidence or {}).get('cues')
        if not cues:
            continue
        print('\nWhat the model noticed:')
        for key, value in cues.items():
            if not value or value in ('unknown', 'none'):
                continue
            if isinstance(value, list):
                value = ', '.join(str(v) for v in value)
            print(f'  {key.replace("_", " "):<18} {value}')

    if pred.rationale:
        print('\nReasoning:\n' + pred.rationale)
    for w in locator.warnings:
        print('\nNote:', w)


print('Choose a photo to locate. Re-run this cell (\u25b6 or Ctrl+Enter) for another.\n')

uploaded = files.upload()
if not uploaded:
    print('Nothing chosen.')
for name in uploaded:
    show(name)
    print('\n' + '\u2500' * 60)
    print('Re-run this cell to try another photo.')


---

### Notes

- **Photos with GPS metadata** are answered exactly and instantly. Phone photos
  often still carry it; anything downloaded from social media almost never does.
- **Nothing is stored.** Images live only in this Colab session, which is
  discarded when you close it. If the reasoning head is on, the image is sent to
  the Anthropic API to be analysed.
- **Measuring it properly:** if you have photos with known coordinates, put them
  in a CSV with `image,lat,lon` columns and run `geolocate eval yourdata.csv`.
  It reports accuracy at the standard thresholds and, importantly, checks whether
  the uncertainty radius is honest.

### Please do not use this to find people

It locates places, not people. Photographs other people took can reveal where
they live or work without their meaning to. Use it on your own photos, on press
imagery you are verifying, or to check what a photo of yours gives away before
you post it.
